# Notebook 03 — Attendance Prediction Model

Feature engineering, model training (Linear Regression, Random Forest, Gradient Boosting), evaluation, and business interpretation.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from src.config import DATA_PROCESSED_DIR
sns.set_theme(style='whitegrid')
print('Setup complete')

## 1. Feature Engineering

In [ ]:
from src.feature_engineering import main as build_features
build_features()

df = pd.read_csv(DATA_PROCESSED_DIR / 'features_attendance.csv', parse_dates=['game_date'])
print(f'Feature dataset: {len(df):,} rows, {len(df.columns)} columns')
print('\nAttendance target stats:')
print(df['actual_attendance'].describe().round(0))

In [ ]:
# Correlation heatmap for numeric features
numeric_feats = ['is_weekend','rivalry_flag','nationally_televised_flag','promotion_flag',
                 'home_team_win_pct_entering_game','away_team_win_pct_entering_game',
                 'home_team_popularity','away_team_popularity','temperature_f',
                 'snow_flag','rolling_att_3','month','actual_attendance']
corr = df[[c for c in numeric_feats if c in df.columns]].corr()
plt.figure(figsize=(12, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Feature Correlation with Attendance Target')
plt.tight_layout()
plt.show()

## 2. Train Attendance Models

In [ ]:
from src.train_attendance_model import main as train_model
train_model()

## 3. Visualize Model Performance

In [ ]:
preds = pd.read_csv(DATA_PROCESSED_DIR / 'attendance_predictions.csv')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(preds['actual_attendance'], preds['predicted_attendance'], alpha=0.4, color='steelblue', s=15)
max_val = max(preds['actual_attendance'].max(), preds['predicted_attendance'].max())
axes[0].plot([0, max_val], [0, max_val], 'r--', lw=1.5)
axes[0].set_xlabel('Actual Attendance')
axes[0].set_ylabel('Predicted Attendance')
axes[0].set_title('Actual vs Predicted Attendance')

errors = preds['predicted_attendance'] - preds['actual_attendance']
axes[1].hist(errors, bins=40, color='salmon', edgecolor='white')
axes[1].axvline(0, color='navy', linestyle='--')
axes[1].set_title('Prediction Error Distribution')
axes[1].set_xlabel('Error (Predicted - Actual)')

plt.tight_layout()
plt.show()

mae  = mean_absolute_error(preds['actual_attendance'], preds['predicted_attendance'])
rmse = np.sqrt(mean_squared_error(preds['actual_attendance'], preds['predicted_attendance']))
r2   = r2_score(preds['actual_attendance'], preds['predicted_attendance'])
print(f'MAE: {mae:.0f} | RMSE: {rmse:.0f} | R²: {r2:.4f}')

## 4. Business Interpretation

In [ ]:
import json
from src.config import MODELS_DIR
with open(MODELS_DIR / 'model_features.json') as f:
    meta = json.load(f)

fi = meta.get('attendance_feature_importances', {})
if fi:
    fi_df = pd.Series(fi).sort_values(ascending=False).head(12)
    fi_df.plot(kind='barh', figsize=(10, 5), color='steelblue')
    plt.title('Top Feature Importances — Attendance Model')
    plt.xlabel('Importance')
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()
    print('\nBusiness interpretation:')
    print('Rolling attendance history, arena capacity, and opponent popularity are primary drivers.')
    print('Weekend flag, rivalry games, and promotion lift contribute incremental demand signals.')